## **1. Install Dependencies**

In [1]:
# Install only required libraries
!pip install -q -U unsloth
!pip install -q -U transformers==4.56.2
!pip install -q -U datasets==4.3.0
!pip install -q -U trl==0.22.2
!pip install -q -U peft accelerate bitsandbytes
!pip install wandb

# Optional
!pip install -q sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 MB 24.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924.2 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.3 MB/s eta 

## **2 — Imports**

In [2]:
# =========================
# 1) Imports
# =========================

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from peft import PeftModel
from dataclasses import dataclass
from typing import Optional, List
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

# =========================
# 2) GPU Check
# =========================

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    raise Exception("Enable GPU in Kaggle: Settings → Accelerator → GPU")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-06-02 16:13:01.415231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780416781.843175      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780416781.954502      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780416782.970060      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780416782.970116      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780416782.970119      58 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA Available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [6]:
# wandb_token = UserSecretsClient().get_secret("wandb")
# login(token=wandb_token)


wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: preranvarshneyji (preranvarshneyji-amity-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## **3. Configuration**

In [7]:
@dataclass
class FineTuneConfig:

    # ---------------- MODEL ----------------
    model_name: str = "google/gemma-3-1b-it"   #####

    load_in_4bit: bool = False
    max_seq_length: int = 512
    dtype: Optional[str] = None

    # ---------------- DATASET ----------------
    dataset_name: str ="VPrerana/medical_meadow_wikidoc_patient_information"
    split: str = "train"

    # ---------------- TRAINING ----------------
    output_dir: str = "medical_outputs"
    per_device_bs: int = 4
    grad_acc_steps: int = 8
    epochs: int = 1
    lr: float = 1e-6  
    warmup_ratio: float = 0.05  ## warmup_steps=10
    logging_steps: int = 10
    packing: bool = False   ########
    seed: int = 3407

    # ---------------- LORA ----------------
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    target_modules: List[str] = None

    use_gc: str = "unsloth"

    # ---------------- SAVE ----------------
    lora_save_path: str = "medical_lora"

    save_merged: bool = True
    merged_save_path: str = "gemma-3-1b-Medical-it"

##  **4. FineTuner Class**

In [10]:
class UnslothFineTuner:

    def __init__(self, cfg: FineTuneConfig):

        self.cfg = cfg

        if self.cfg.target_modules is None:
            self.cfg.target_modules = [
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj"
            ]

        self.model = None
        self.tokenizer = None

    # ==========================================================
    # LOAD MODEL
    # ==========================================================
    def load_model(self):

        print(f"Loading model: {self.cfg.model_name}")

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.cfg.model_name,
            max_seq_length=self.cfg.max_seq_length,
            dtype=self.cfg.dtype,
            load_in_4bit=self.cfg.load_in_4bit,
        )

        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=self.cfg.lora_r,
            target_modules=self.cfg.target_modules,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            use_gradient_checkpointing=self.cfg.use_gc,
            random_state=self.cfg.seed,
        )

        print("\nTrainable Parameters:\n")
        self.model.print_trainable_parameters()

    # ==========================================================
    # LOAD DATASET
    # ==========================================================
    def load_dataset(self):

        print(f"\nLoading Dataset: {self.cfg.dataset_name}")

        try:
            ds = load_dataset(
                self.cfg.dataset_name,
                split=self.cfg.split,
            )

        except Exception:

            ds = load_dataset(
                path=self.cfg.dataset_name,
                split=self.cfg.split,
            )

        ds = ds.shuffle(seed=self.cfg.seed)
        print(ds)
        print("\nColumns:", ds.column_names)

        return ds

    # ==========================================================
    # EOS
    # ==========================================================
    def _eos(self):
        return self.tokenizer.eos_token or "</s>"

    # ==========================================================
    # PROMPT TEMPLATE
    # ==========================================================
    def _medical_prompt(self):

        return """### Instruction:
{instruction}

### Input:
{input}

### Response:

### End
"""

    # ==========================================================
    # FORMAT ALPACA STYLE
    # ==========================================================
    def _format_alpaca(self, batch):
        EOS = self._eos()
        prompt_template = self._medical_prompt()
        texts = []
        inputs = batch.get(
        "input",
        [""] * len(batch["instruction"])
        )
    
        for ins, inp, out in zip(
            batch["instruction"],
            inputs,
            batch["output"]
        ):
    
            text = (
                prompt_template.format(
                    instruction=ins or "",
                    input=inp or "",
                )
                + (out or "")
                + EOS
            )
    
            texts.append(text)
    
        return {"text": texts}

    # ==========================================================
    # FORMAT QUESTION ANSWER
    # ==========================================================
    def _format_question_answer(self, batch):
        EOS = self._eos()
        texts = []
        
        for q, a in zip(batch["question"], batch["answer"]):
            text = f"""Instruction:
Answer the following medical question.

### Input:
{question}

### Response:
"""

            texts.append(text + EOS)

        return {"text": texts}

    # ==========================================================
    # FORMAT TEXT
    # ==========================================================
    def _format_text(self, batch):

        EOS = self._eos()

        return {
            "text": [
                (t or "") + EOS
                for t in batch["text"]
            ]
        }

    # ==========================================================
    # FORMAT CHAT
    # ==========================================================
    def _format_sharegpt(self, batch):

        EOS = self._eos()

        texts = []

        conv_key = None

        if "conversations" in batch:
            conv_key = "conversations"

        elif "messages" in batch:
            conv_key = "messages"

        for conv in batch[conv_key]:

            if conv is None:
                texts.append("" + EOS)
                continue

            chat = ""

            for t in conv:

                role = ""
                content = ""

                if "from" in t and "value" in t:
                    role = t["from"]
                    content = t["value"]

                elif "role" in t and "content" in t:
                    role = t["role"]
                    content = t["content"]

                if role in ["human", "user"]:
                    chat += f"### User:\n{content}\n\n"

                else:
                    chat += f"### Assistant:\n{content}\n\n"

            texts.append(chat + EOS)

        return {"text": texts}

    # ==========================================================
    # FORMAT DATASET
    # ==========================================================
    def format_dataset(self, ds):

        print("\nInferring Dataset Format...")

        cols = set(ds.column_names)

        # instruction dataset
        if {"instruction", "input", "output"}.issubset(cols):

            print("Detected: Instruction Dataset")

            return ds.map(
                self._format_alpaca,
                batched=True,
                remove_columns=ds.column_names,
            )

        # question answer dataset
        if {"question", "answer"}.issubset(cols):

            print("Detected: Question Answer Dataset")

            return ds.map(
                self._format_question_answer,
                batched=True,
                remove_columns=ds.column_names,
            )

        # text dataset
        if "text" in cols:

            print("Detected: Text Dataset")

            return ds.map(
                self._format_text,
                batched=True,
                remove_columns=ds.column_names,
            )

        # chat dataset
        if "conversations" in cols or "messages" in cols:

            print("Detected: Chat Dataset")

            return ds.map(
                self._format_sharegpt,
                batched=True,
                remove_columns=ds.column_names,
            )

        raise ValueError(
            f"Could not infer dataset format. Columns found: {sorted(list(cols))}"
        )

    # ==========================================================
    # TRAIN
    # ==========================================================
    def train(self, ds):

        print("\nStarting Training...")

        wandb.init(
        project="Google-gemma-finetuning-medical",
        name="gemma-5k-run"
       )

        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=ds,
            dataset_text_field="text",
            packing=self.cfg.packing,

            args=SFTConfig(
                per_device_train_batch_size=self.cfg.per_device_bs,
                gradient_accumulation_steps=self.cfg.grad_acc_steps,
                num_train_epochs=self.cfg.epochs,
                learning_rate=self.cfg.lr,
                warmup_ratio=self.cfg.warmup_ratio,
                logging_steps=self.cfg.logging_steps,
                output_dir=self.cfg.output_dir,
                optim="paged_adamw_32bit",   ## "adamw_8bit",
                report_to="wandb",
                seed=self.cfg.seed,
                max_steps=400,

                fp16=not torch.cuda.is_bf16_supported(),
                bf16=torch.cuda.is_bf16_supported(),
            ),
        )

        trainer.train()

    # ==========================================================
    # SAVE MODEL
    # ==========================================================
    def save(self):

        print("\nSaving LoRA Adapters...")

        self.model.save_pretrained(self.cfg.lora_save_path)
        self.tokenizer.save_pretrained(self.cfg.lora_save_path)

        if self.cfg.save_merged:
            print("\nSaving Merged Model...")

            merged_model = self.model.merge_and_unload()
            merged_model.save_pretrained(
                self.cfg.merged_save_path,
                safe_serialization=True
            )

            self.tokenizer.save_pretrained(
                self.cfg.merged_save_path
            )

    # ==========================================================
    # RUN COMPLETE PIPELINE
    # ==========================================================
    def run(self):

        self.load_model()
        raw_ds = self.load_dataset()

        print(raw_ds[0])
        print(raw_ds[10])
        print(raw_ds[100])
    
        ds = self.format_dataset(raw_ds)
        print("\nSample Formatted Text:\n")
        print(ds["text"][0][:1000])

        self.train(ds)
        self.save()

        print("\nTraining Complete!")

## 5. Start **Training**

In [11]:
cfg = FineTuneConfig()

trainer = UnslothFineTuner(cfg)

trainer.run()

Loading model: google/gemma-3-1b-it
==((====))==  Unsloth 2026.5.10: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.

Trainable Parameters:

trainable params: 6,522,880 || all params: 1,006,408,832 || trainable%: 0.6481

Loading Dataset: VPrerana/medical_meadow_wikidoc_patient_information
Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5942
})

Columns: ['instruction', 'input', 'output']
{'instruction': 'Answer this question truthfully', 'input': 'W

Map:   0%|          | 0/5942 [00:00<?, ? examples/s]


Sample Formatted Text:

### Instruction:
Answer this question truthfully

### Input:
What are the symptoms of Gallstone disease?

### Response:

### End
As gallstones move into the bile ducts and create blockage, pressure increases in the gallbladder and one or more symptoms may occur. Symptoms of blocked bile ducts are often called a gallbladder “attack” because they occur suddenly. Gallbladder attacks often follow fatty meals, and they may occur during the night. A typical attack can cause;
Steady pain in the right upper abdomen that increases rapidly and lasts from 30 minutes to several hours Pain in the back between the shoulder blades Pain under the right shoulder
Notify your doctor if you think you have experienced a gallbladder attack. Although these attacks often pass as gallstones move, your gallbladder can become infected and rupture if a blockage remains.
People with any of the following symptoms should see a doctor immediately:
Prolonged pain —more than 5 hours Nausea and 

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/5942 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,942 | Num Epochs = 3 | Total steps = 400
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 6,522,880 of 1,006,408,832 (0.65% trained)


Step,Training Loss
10,3.702000
20,3.633100
30,3.765400
40,3.674000
50,3.625900
60,3.555900
70,3.593700
80,3.535900
90,3.493200
100,3.472200


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!

Saving LoRA Adapters...

Saving Merged Model...

Training Complete!


## **** **wandb plot** 

In [12]:
# Wandb initialization for tracking
# wandb.login(api_key=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)
run = wandb.init(project='Google-gemma-finetuning-medical', job_type="training", anonymous="allow")

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,▆▆▄▅██▇▅▄▅▅▃▂▃▃▁▂▃▂▃▆▂▃▃▃▃▃▃▂▂▃▃▃▃▃▄▆▅▄▄
train/learning_rate,▄████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,▇▇█▇▇▆▇▆▆▆▄▄▄▅▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▂▂▂▁▂▁
total_flos,6765308571232512.0
train/epoch,2.15074
train/global_step,400
train/grad_norm,4.14027
train/learning_rate,0.0
train/loss,2.9412


## 6. After training **Load finetuned model for Inference**

In [13]:
from unsloth import FastLanguageModel
import torch

model_path="gemma-3-1b-Medical-it"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=2048,
    load_in_4bit=False,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.5.10: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((11

## 7. **Inferencing the model**

In [59]:
def generate_medical_response(question):

    prompt = f"""
### Instruction:
Provide accurate medical information.

### Input:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to("cuda")

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            # min_new_tokens=80,
            do_sample=False,
            # temperature=0.7,
            repetition_penalty=1.00,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

     # Remove disclaimer section
    disclaimer_keywords = [
        "### Disclaimer:",
        "**Disclaimer:**",
        "Important Disclaimer:",
        "IMPORTANT DISCLAIMER:",
        "Disclaimer:",
        "Important disclaimer:"
    ]

    for keyword in disclaimer_keywords:
        if keyword in response:
            response = response.split(keyword)[0].strip()


    response = response.split("### Response:")[-1]
    response = response.split("### End")[0].strip()

    return response



## **8. Test the model**

In [15]:
question = """
A 3-year-old child presents with barking cough, inspiratory stridor, and hoarseness.Symptoms worsen at night.

Options:
A. Asthma
B. Bronchiolitis
C. Croup
D. Pneumonia
"""

response = generate_medical_response(question)

print(response)

B. Bronchiolitis

### Explanation:

Bronchiolitis is a common respiratory infection in young children, characterized by barking cough, inspiratory stridor, and hoarseness. It often worsens at night.

Here's why the other options are less likely:

*   **Asthma:** While asthma can cause wheezing and coughing, it typically doesn't present with a stridor or hoarseness.
*   **Croup:** Croup is typically associated with a barking cough and stridor, often triggered by a cold.
*   **Pneumonia:** Pneumonia is a more serious lung infection, and while it can cause these symptoms, it's less likely to present with a barking cough and inspiratory stridor as the primary symptoms.


In [17]:
question = """
A 67-year-old man comes to the physician because of progressive shortness of breath and swelling in his legs over the past 6 months. He has a history of hypertension and myocardial infarction. Physical examination shows jugular venous distention, bilateral pitting edema, and crackles at both lung bases. Echocardiography reveals reduced ejection fraction.

Options:
A. Right ventricular hypertrophy
B. Systolic heart failure
C. Restrictive cardiomyopathy
D. Mitral valve prolapse
E. Cardiac tamponade
"""

response = generate_medical_response(question)

print(response)

B. Systolic heart failure

### Explanation:

Here's why:

*   **Clinical Presentation:** The patient's symptoms (progressive shortness of breath, leg swelling, jugular venous distention, bilateral pitting edema, crackles) are highly suggestive of systolic heart failure.
*   **Echocardiography Findings:** Reduced ejection fraction confirms heart failure.
*   **History:** The patient's history of hypertension and myocardial infarction significantly increases the risk of heart failure.
*   **Other Options:**
    *   A. Right ventricular hypertrophy: This is less likely given the patient's symptoms and echocardiogram findings.
    *   C. Restrictive cardiomyopathy: This is a possibility, but the echocardiogram results point more strongly towards heart failure.
    *   D. Mitral valve prolapse: This is a common condition, but it doesn't explain the severity of the symptoms and the presence of significant edema and crackles.
    *   E. Cardiac tamponade: This is a life-threatening emergency,

In [18]:
question = """
A 58-year-old man with chronic smoking history presents with hematuria and flank pain. Imaging reveals a renal mass. Histology shows clear cells.

Options:
A. Wilms tumor
B. Transitional cell carcinoma
C. Renal cell carcinoma
D. Oncocytoma
E. Angiomyolipoma
"""

response = generate_medical_response(question)

print(response)

C. Renal cell carcinoma

### Explanation:

The patient's presentation – hematuria, flank pain, and a renal mass on imaging – strongly suggests renal cell carcinoma. Here's why:

*   **Renal cell carcinoma:** This is the most common type of kidney cancer, and it frequently presents with hematuria (blood in the urine), flank pain, and a renal mass on imaging. Clear cells are a characteristic feature of renal cell carcinoma.

Let's look at why the other options are less likely:

*   **Wilms tumor:** This is a childhood kidney cancer, typically presenting with more acute symptoms and often involving the lower abdomen.
*   **Transitional cell carcinoma:** This is a rare type of skin cancer that can occur in the kidneys, but it's less common than renal cell carcinoma.
*   **Oncocytoma:** These are benign tumors that can occur in the kidneys, but they are typically smaller and less aggressive than renal cell carcinoma.
*   **Angiomyolipoma:** These are benign tumors that can occur in the kidn

In [19]:
question = """
A 30-year-old woman presents with heat intolerance, weight loss despite increased appetite, and palpitations. On examination, she has exophthalmos and diffuse enlargement of the thyroid gland.

Options:
A. Hashimoto thyroiditis
B. Graves disease
C. Subacute thyroiditis
D. Toxic multinodular goiter
E. Thyroid adenoma
"""

response = generate_medical_response(question)

print(response)

B. Graves disease

### Explanation:

Here's why Graves disease is the most likely diagnosis based on the provided information:

*   **Exophthalmos:** This is a hallmark symptom of Graves disease, indicating an enlarged thyroid gland.
*   **Diffuse thyroid enlargement:** Graves disease is a common cause of this, as the thyroid gland becomes enlarged due to the production of thyroid hormones.
*   **Heat intolerance, weight loss despite increased appetite:** These are classic symptoms of Graves disease, often associated with hyperthyroidism.
*   **Palpitations:** Hyperthyroidism can cause palpitations.

Let's look at why the other options are less likely:

*   **A. Hashimoto thyroiditis:** While Hashimoto can cause exophthalmos and thyroid enlargement, it typically doesn't present with the degree of diffuse enlargement and the specific constellation of symptoms (heat intolerance, weight loss, palpitations) as strongly as Graves disease.
*   **C. Subacute thyroiditis:** This is a less comm

In [20]:
question = """
A 24-year-old woman presents with fatigue, pallor, and spoon-shaped nails. Laboratory studies show:
- Hemoglobin: low
- Mean corpuscular volume: low
- Ferritin: low

Which of the following is the most likely diagnosis?

Options:
A. Vitamin B12 deficiency
B. Iron deficiency anemia
C. Aplastic anemia
D. Sickle cell anemia
E. Autoimmune hemolytic anemia
"""

response = generate_medical_response(question)

print(response)

B. Iron deficiency anemia

### Explanation:

Here's why this is the most likely diagnosis:

*   **Symptoms:** The patient presents with fatigue, pallor (paleness), and spoon-shaped nails (koilonychia), which are classic symptoms of iron deficiency anemia.
*   **Laboratory Findings:** The low hemoglobin, low mean corpuscular volume (MCV), and low ferritin levels are all indicative of iron deficiency.  Low ferritin is particularly important because it reflects the body's reduced iron stores.
*   **Why other options are less likely:**
    *   **Vitamin B12 deficiency:** While vitamin B12 deficiency can cause anemia, it typically presents with neurological symptoms (numbness, tingling) and can be accompanied by other abnormalities. The patient's presentation is less suggestive of these.
    *   **Aplastic anemia:** Aplastic anemia is a bone marrow failure disorder, and the lab findings are not consistent with this condition.
    *   **Sickle cell anemia:** Sickle cell anemia is characteriz

In [21]:
question = """
My daughter is 8 she woke up in the middle of the night holding her stomach saying it hurts. What is the actual problem she might be facing?
"""

response = generate_medical_response(question)

print(response)

Okay, let's talk about what might be causing your daughter's stomach pain. It's important to understand that I am an AI and cannot provide a diagnosis. However, here's a breakdown of potential causes, ranging from more common to less common, and what you should do:

**Possible Causes:**

1.  **Gas:** This is a very common cause of stomach pain, especially in children.  Gas buildup can cause bloating, discomfort, and pain.
2.  **Constipation:**  If she's not passing stool regularly, it can lead to abdominal discomfort and pain.
3.  **Viral Gastroenteritis (Stomach Flu):**  This is a common illness that can cause nausea, vomiting, diarrhea, and abdominal pain.
4.  **Food Intolerance/Allergy:**  Certain foods can trigger reactions in some children, leading to stomach pain, bloating, and other symptoms.
5.  **Overeating:**  Eating too much, especially quickly, can strain the digestive system.
6.  **Muscle Strain:**  Sometimes, abdominal pain can be caused by straining during bowel movement

In [22]:
question = """
A 26-year-old woman presents with fatigue, dark urine, and jaundice. Laboratory studies show:
- Hemoglobin: 8.1 g/dL
- Reticulocyte count: elevated
- LDH: elevated
- Haptoglobin: decreased
Direct Coombs test is positive.

What is the most likely diagnosis?

Options:
A. Hereditary spherocytosis
B. G6PD deficiency
C. Autoimmune hemolytic anemia
D. Sickle cell disease
E. Iron deficiency anemia
"""

response = generate_medical_response(question)

print(response)

C. Autoimmune hemolytic anemia

### Explanation:

Here's why this is the most likely diagnosis:

*   **Clinical Presentation:** The patient presents with fatigue, dark urine, and jaundice, which are classic symptoms of hemolytic anemia.
*   **Laboratory Findings:**
    *   **Elevated Hemoglobin and Reticulocyte count:** These indicate anemia, and the elevated reticulocyte count suggests the bone marrow is trying to compensate for the anemia.
    *   **Elevated LDH:** LDH is released into the bloodstream when red blood cells are damaged. Elevated levels suggest hemolysis.
    *   **Decreased Haptoglobin:** Haptoglobin is a protein that binds to free hemoglobin. Its decreased level indicates that the body is losing hemoglobin to the urine.
    *   **Direct Coombs test positive:** This indicates antibodies are attacking red blood cells, which is a hallmark of autoimmune hemolytic anemia.

Let's look at why the other options are less likely:

*   **A. Hereditary spherocytosis:** This is a 

In [23]:
question = """
A patient presents with severe anxiety and rapid breathing. Arterial blood gas shows:
- pH: 7.52
- PaCO₂: 28 mmHg
- HCO₃⁻: 23 mEq/L

What is the primary acid-base disturbance?

Options:
A. Metabolic acidosis
B. Metabolic alkalosis
C. Respiratory acidosis
D. Respiratory alkalosis
E. Mixed metabolic acidosis and respiratory alkalosis
"""

response = generate_medical_response(question)

print(response)

C. Respiratory acidosis

### Explanation:

The patient's arterial blood gas results demonstrate a significant decrease in pH (7.52) and an elevated PaCO₂ (28 mmHg).  This combination strongly indicates respiratory acidosis.

Here's why:

*   **Low pH:** A lower pH indicates the blood is more acidic.
*   **Elevated PaCO₂:** PaCO₂ is the measure of carbon dioxide in the blood.  Increased CO₂ is a compensatory mechanism to help remove excess acid from the body.
*   **Low HCO₃⁻:**  Low bicarbonate levels indicate a loss of bicarbonate, which is a buffer in the blood.

Let's look at why the other options are incorrect:

*   **A. Metabolic acidosis:** Metabolic acidosis is characterized by a low pH and elevated bicarbonate levels.
*   **B. Metabolic alkalosis:** Metabolic alkalosis is characterized by a high pH and low bicarbonate levels.
*   **D. Respiratory alkalosis:** Respiratory alkalosis is characterized by a high pH and low bicarbonate levels.
*   **E. Mixed metabolic acidosis and res

In [24]:
question = """
A 4-year-old child presents with fatigue, recurrent infections, and easy bruising. Bone marrow biopsy reveals lymphoblasts positive for TdT.

What is the most likely diagnosis?

Options:
A. Acute lymphoblastic leukemia
B. Acute myeloid leukemia
C. Chronic lymphocytic leukemia
D. Hodgkin lymphoma
E. Multiple myeloma
"""

response = generate_medical_response(question)

print(response)

A. Acute lymphoblastic leukemia

**Explanation:**

*   **Lymphoblasts positive for TdT:** This is a highly characteristic finding in acute lymphoblastic leukemia (ALL). TdT (transcription factor T-delta) is a gene that is frequently mutated in ALL.
*   **Fatigue, recurrent infections, and easy bruising:** These are common symptoms of ALL, particularly in children.
*   **Bone marrow biopsy reveals lymphoblasts:** This confirms the presence of leukemia cells in the bone marrow.
*   **Acute lymphoblastic leukemia** is the most common type of leukemia in children and is characterized by the presence of TdT-positive lymphoblasts.

**Why the other options are less likely:**

*   **B. Acute myeloid leukemia:** While ALL is a possibility, it's less common in children than acute lymphoblastic leukemia.
*   **C. Chronic lymphocytic leukemia:** This is characterized by persistent, slow-growing lymphocytes, not the rapid proliferation of lymphoblasts seen in ALL.
*   **D. Hodgkin lymphoma:** This 

In [25]:
question = """
A patient with chronic alcoholism presents with confusion and seizures. Laboratory studies reveal:
- Sodium: 112 mEq/L
- Serum osmolality: low
- Urine sodium: elevated

Rapid correction of this condition may lead to which complication?

Options:
A. Cerebral edema
B. Central pontine myelinolysis
C. Subarachnoid hemorrhage
D. Wernicke encephalopathy
E. Brain abscess

"""

response = generate_medical_response(question)

print(response)

B. Central pontine myelinolysis

**Explanation:**

*   **Central Pontine Myelolysis:** This is the most likely complication of rapid correction of chronic alcoholism-induced confusion and seizures. Rapid correction involves the administration of hypertonic saline solutions, which disrupt the myelin sheath surrounding nerve fibers in the pons. This leads to the breakdown of myelin, causing a rapid loss of nerve function and, ultimately, seizures.

Let's look at why the other options are less likely:

*   **A. Cerebral edema:** While cerebral edema can occur with severe dehydration, it's not directly caused by rapid correction of the electrolyte imbalance.
*   **C. Subarachnoid hemorrhage:** Subarachnoid hemorrhage is a serious condition that can occur with any neurological event, but it's not specifically linked to the rapid correction of the electrolyte imbalance.
*   **D. Wernicke encephalopathy:** Wernicke's encephalopathy is caused by a deficiency of thiamine, not by rapid correctio

In [26]:
question = """
A 45-year-old obese woman presents with right upper quadrant pain after fatty meals. Ultrasound reveals gallstones. She suddenly develops fever, jaundice, and hypotension.

What is the most likely diagnosis?

Options:
A. Acute pancreatitis
B. Acute hepatitis
C. Ascending cholangitis
D. Acute cholecystitis
E. Peptic ulcer disease
"""

response = generate_medical_response(question)

print(response)

C. Ascending cholangitis

**Explanation:**

Here's why this is the most likely diagnosis:

*   **Ascending cholangitis** is a serious infection of the bile ducts. It's a significant concern given the patient's symptoms:
    *   **Right upper quadrant pain after fatty meals:** This is a classic presentation of cholangitis.
    *   **Fever, jaundice, and hypotension:** These are all signs of systemic inflammation and infection, characteristic of cholangitis.
    *   **Gallstones:** While gallstones can contribute to cholangitis, they are not the primary cause.
    *   **The sudden onset of these symptoms after fatty meals is highly suggestive of an acute infection.**

Let's look at why the other options are less likely:

*   **A. Acute pancreatitis:** While pancreatitis can cause abdominal pain and fever, it typically doesn't present with the same combination of jaundice, hypotension, and a sudden onset of these symptoms.
*   **B. Acute hepatitis:** Acute hepatitis is usually associated 

In [27]:
question = """
A 29-year-old woman presents with weight loss, palpitations, heat intolerance, and exophthalmos. Laboratory testing shows decreased TSH and elevated T3/T4 levels.

What is the most likely diagnosis?

Options:
A. Hashimoto thyroiditis
B. Graves disease
C. Subacute thyroiditis
D. Toxic multinodular goiter
E. Pituitary adenoma
"""

response = generate_medical_response(question)

print(response)

B. Graves disease

### Explanation:

Here's why Graves disease is the most likely diagnosis based on the provided information:

*   **Weight loss and palpitations:** These are classic symptoms of Graves disease.
*   **Heat intolerance:** This is a common symptom of hyperthyroidism, which is a hallmark of Graves disease.
*   **Exophthalmos:** This is a significant finding in Graves disease, indicating an enlarged thyroid gland.
*   **Decreased TSH and elevated T3/T4 levels:** These are consistent with hyperthyroidism, a key feature of Graves disease.

Let's look at why the other options are less likely:

*   **A. Hashimoto thyroiditis:** While Hashimoto can cause symptoms like weight loss and palpitations, it typically doesn't present with exophthalmos or elevated T3/T4 levels.
*   **C. Subacute thyroiditis:** This can cause similar symptoms, but it's less likely to be associated with the specific combination of symptoms described (exophthalmos, heat intolerance, and elevated T3/T4).
* 

In [28]:
question = """
A 35-year-old HIV-positive man presents with headache, fever, and confusion. MRI reveals multiple ring-enhancing lesions in the brain. CD4 count is 40 cells/mm³.
Which organism is the most likely cause?

Options:
A. Cryptococcus neoformans
B. Toxoplasma gondii
C. JC virus
D. Cytomegalovirus
E. Mycobacterium tuberculosis
"""

response = generate_medical_response(question)

print(response)

A. Cryptococcus neoformans

### Explanation:

Here's why:

*   **Cryptococcus neoformans** is a common opportunistic fungal infection in immunocompromised individuals, particularly HIV-positive patients. It often presents with headache, fever, and confusion, and MRI findings of ring-enhancing lesions are highly characteristic of cryptococcal meningitis.

Let's look at why the other options are less likely:

*   **Toxoplasma gondii:** While possible, it's less common in HIV-positive individuals, and typically presents with more generalized symptoms like fever and lymphadenopathy.
*   **JC virus:** JC virus is a retrovirus and is not associated with the clinical presentation described.
*   **Cytomegalovirus:** Cytomegalovirus can cause meningitis, but it's less frequently associated with the specific clinical picture of ring-enhancing lesions and confusion seen in this case.
*   **Mycobacterium tuberculosis:** Tuberculosis is a serious infection, but it's less likely to present with the 

In [29]:
question = """
A 63-year-old man with atrial fibrillation is started on a medication. Two weeks later, he develops pulmonary fibrosis, hepatotoxicity, and blue-gray skin discoloration.
Which drug is most likely responsible?

Options:
A. Digoxin
B. Lidocaine
C. Procainamide
D. Amiodarone
E. Verapamil

"""

response = generate_medical_response(question)

print(response)

D. Amiodarone

**Explanation:**

Amiodarone is the most likely drug responsible for the patient's symptoms. Here's why:

*   **Pulmonary Fibrosis:** Amiodarone is a potent antiarrhythmic drug that has been strongly linked to pulmonary fibrosis, a chronic lung disease.
*   **Hepatic Toxicity:** Amiodarone can cause significant liver damage, leading to hepatotoxicity.
*   **Blue-Gray Skin Discoloration:** This is a characteristic symptom of amiodarone-induced liver toxicity, often appearing as a bluish-gray discoloration of the skin.
*   **Other Options:**
    *   Digoxin: Primarily used for heart rate control, not typically associated with pulmonary fibrosis or liver toxicity.
    *   Lidocaine: A local anesthetic, not a medication for cardiac arrhythmias or fibrosis.
    *   Procainamide: A vasodilator used for angina, not associated with the described symptoms.
    *   Verapamil: A calcium channel blocker used for hypertension and angina, not linked to the observed symptoms.


In [30]:
question = """
A newborn develops bilious vomiting shortly after birth. Physical examination reveals abdominal distention. X-ray shows a “double bubble” sign. The infant also has upward-slanting palpebral fissures and a flat facial profile.
What is the most likely diagnosis?

Options:
A. Hirschsprung disease
B. Duodenal atresia
C. Pyloric stenosis
D. Intussusception
E. Meconium ileus
"""

response = generate_medical_response(question)

print(response)

E. Meconium ileus

### Explanation:

Meconium ileus is a rare but serious condition that can occur shortly after birth. It is characterized by the accumulation of meconium (the waste material from the fetus's gut) in the intestines, leading to a buildup of pressure and a characteristic "double bubble" sign on abdominal X-rays. The other options are less likely to present with this specific combination of findings:

*   **Hirschsprung disease:** Typically presents with a neurological deficit and bowel obstruction.
*   **Duodenal atresia:** Involves a blockage of the duodenum, often with associated vomiting.
*   **Pyloric stenosis:** A narrowing of the pyloric sphincter, leading to vomiting and abdominal distention.
*   **Intussusception:** A telescoping of the intestines, causing vomiting and abdominal pain.

Therefore, the combination of bilious vomiting, abdominal distention, a "double bubble" sign, upward-slanting palpebral fissures, and a flat facial profile strongly suggests meconi

In [31]:
question = """
A 71-year-old woman suddenly develops right facial droop, weakness of the right arm and leg, and difficulty speaking. Physical examination shows expressive aphasia.

Which artery is most likely occluded?

Options:
A. Anterior cerebral artery
B. Middle cerebral artery
C. Posterior cerebral artery
D. Basilar artery
E. Posterior inferior cerebellar artery
"""

response = generate_medical_response(question)

print(response)

E. Posterior inferior cerebellar artery

**Explanation:**

The patient presents with a constellation of symptoms strongly suggestive of a stroke. The combination of right facial droop, weakness of the right arm and leg, and difficulty speaking points towards a vascular event affecting the brainstem.

Here's why the other options are less likely:

*   **A. Anterior cerebral artery:** This artery supplies the motor cortex and is less likely to cause such a rapid and severe presentation.
*   **B. Middle cerebral artery:** While it can be affected by stroke, it's less likely to cause the rapid and widespread symptoms described.
*   **C. Posterior cerebral artery:** This artery is primarily involved in visual processing and is less directly implicated in the symptoms of a stroke affecting speech and motor function.
*   **D. Basilar artery:** This artery is primarily involved in hearing and balance and is not typically associated with the symptoms of a stroke.

**Important Note:** This infor

In [32]:
question = """
A 58-year-old man comes to the emergency department because of severe chest pain radiating to his left arm for the past 2 hours. ECG shows ST-segment elevation in leads II, III, and aVF. Blood pressure is 82/58 mm Hg. Physical examination reveals elevated jugular venous pressure and clear lungs.

What is the most appropriate next step in management?

Options:
A. Administer intravenous furosemide
B. Begin high-dose beta blockers
C. Administer nitrates
D. Perform right-sided ECG leads
E. Immediate pericardiocentesis
"""

response = generate_medical_response(question)

print(response)

C. Administer nitrates

**Explanation:**

Here's why this is the most appropriate next step:

*   **ST-segment elevation in leads II, III, and aVF:** This is a critical finding suggesting an acute myocardial infarction (MI), specifically an inferior wall MI.
*   **Severe chest pain radiating to the left arm:** This is a classic symptom of MI.
*   **Elevated jugular venous pressure:** This indicates possible cardiogenic shock, which can worsen the ischemic process.
*   **Clear lungs:** This is reassuring, but doesn't address the underlying cause of the MI.

**Nitrates** are the most appropriate initial intervention for acute MI. Nitrates help to reduce preload and afterload, which can improve myocardial perfusion and reduce myocardial damage. They are generally the first-line treatment for acute MI.

Let's look at why the other options are less appropriate:

*   **A. Administer intravenous furosemide:** Furosemide is a diuretic used to manage fluid overload, not for acute MI.
*   **B. B

In [33]:
question = """
Why should alcohol be avoided while taking metronidazole?
"""

response = generate_medical_response(question)

print(response)

Alcohol can interact with metronidazole and increase the risk of side effects.

### Additional Information:
*   Metronidazole is an antibiotic that is used to treat infections.
*   Alcohol can affect the liver, and it can cause liver damage.
*   Alcohol can also affect the nervous system, and it can cause dizziness, drowsiness, and confusion.
*   The combination of metronidazole and alcohol can increase the risk of serious side effects.


In [34]:
question = """
A patient suddenly becomes unconscious and has no pulse.What should be done immediately?
"""

response = generate_medical_response(question)

print(response)

**Immediate Actions:**

1.  **Call for Emergency Medical Services (EMS):** This is the most crucial step.  Do not delay.  Provide the location and a brief description of the situation.
2.  **Check for Breathing:**  Observe the patient for any signs of breathing (chest rise and fall, movement).  If there is no breathing, begin rescue breathing (mouth-to-mouth resuscitation) or CPR.
3.  **Check for a Pulse:**  If you are trained to do so, check for a pulse.  If you are not trained, do not waste time.
4.  **Start CPR (Cardiopulmonary Resuscitation):** If no breathing or pulse, begin CPR immediately.  This involves chest compressions and rescue breaths.
5.  **Assess for Circulation:**  Look for signs of circulation (skin color, temperature, capillary refill).
6.  **Administer Oxygen (if available):** If available, administer supplemental oxygen.
7.  **Monitor Vital Signs:** Continuously monitor the patient's vital signs (heart rate, blood pressure, respiratory rate, oxygen saturation) unti

In [35]:
question = """
Explain what hypertension is in simple language.
"""

response = generate_medical_response(question)

print(response)

Hypertension is simply defined as high blood pressure. It means your blood pressure is consistently higher than normal.

### Additional Information:
*   **Normal Blood Pressure:** A normal blood pressure is generally considered to be below 120/80 mmHg.
*   **Symptoms:** High blood pressure can cause symptoms like headaches, dizziness, and shortness of breath.
*   **Risk Factors:** Several factors can contribute to hypertension, including age, family history, high sodium intake, and lack of physical activity.


In [36]:
question = """
A patient presents with:
- obesity
- hypertension
- elevated fasting glucose
- high triglycerides

What syndrome is most likely present?
"""

response = generate_medical_response(question)

print(response)

The most likely syndrome is Type 2 Diabetes Mellitus.

### Explanation:
The patient's symptoms – obesity, hypertension, elevated fasting glucose, and high triglycerides – are all highly suggestive of Type 2 Diabetes Mellitus. These conditions are commonly associated with insulin resistance and impaired glucose metabolism.

### Additional Notes:
- Obesity is a major risk factor for Type 2 Diabetes.
- Hypertension is a common complication of diabetes.
- Elevated fasting glucose indicates hyperglycemia, a hallmark of the disease.
- High triglycerides are often associated with insulin resistance and metabolic syndrome.
- Type 2 Diabetes Mellitus is a chronic condition that requires ongoing management.


In [37]:
question = """
What are the common side effects of ibuprofen?
"""

response = generate_medical_response(question)

print(response)

Ibuprofen is a common medication that can cause a variety of side effects. Here's a breakdown of the most common ones:

*   **Gastrointestinal Issues:** This is the most frequent side effect. It can include:
    *   Heartburn
    *   Indigestion
    *   Nausea
    *   Stomach pain
    *   Bloating
    *   Diarrhea
    *   Constipation

*   **Dizziness:** Some people may experience dizziness, especially when standing up quickly.

*   **Headache:**  Although ibuprofen is often used to treat headaches, it can sometimes cause a headache.

*   **Fluid Retention:**  This can lead to swelling in the ankles and feet.

*   **Skin Reactions:**  Rarely, ibuprofen can cause skin rashes or itching.

*   **Other Possible Side Effects:**
    *   Increased blood pressure
    *   Increased risk of bleeding (especially if taking blood thinners)
    *   Tinnitus (ringing in the ears)


In [38]:
question = """
A patient complains of fatigue and dizziness.
Laboratory results show:
- low hemoglobin
- low ferritin
- microcytic hypochromic RBCs

What is the most likely diagnosis?
"""

response = generate_medical_response(question)

print(response)

The most likely diagnosis is anemia.

### Explanation:
The patient's symptoms of fatigue and dizziness, combined with the laboratory results of low hemoglobin, low ferritin, and microcytic hypochromic RBCs, strongly suggest anemia.

Here's a breakdown of why this is the most likely diagnosis:

*   **Anemia:** This condition involves a deficiency of red blood cells or hemoglobin, leading to reduced oxygen-carrying capacity.
*   **Low Hemoglobin:** This is a hallmark of anemia, indicating a reduced number of red blood cells.
*   **Low Ferritin:** Ferritin is a protein that stores iron. Low ferritin levels indicate that the body is not efficiently utilizing iron, which can be a consequence of anemia.
*   **Microcytic hypochromic RBCs:** This describes the size and color of the red blood cells. Microcytic means the red blood cells are smaller than normal, and hypochromic means they are pale in color. These are all consistent with anemia.


In [39]:
question = """
A patient develops well-demarcated erythematous plaques with silvery scales on the elbows and knees.What is the most likely diagnosis?
"""

response = generate_medical_response(question)

print(response)

The most likely diagnosis is **Psoriatic Arthritis**.

### Explanation:
Psoriatic arthritis is characterized by the presence of well-demarcated erythematous plaques, silvery scales, and joint involvement. The patient's presentation aligns with this diagnosis.


In [40]:
question = """
A 16-year-old boy presents with abdominal pain that started nearthe umbilicus and later migrated to the right lower quadrant.He has nausea and fever.
What is the most likely diagnosis that boy could done?
"""

response = generate_medical_response(question)

print(response)

The most likely diagnosis is appendicitis.

### Explanation:
Appendicitis is a common condition in adolescents, and the symptoms described – abdominal pain that starts near the umbilicus and migrates to the right lower quadrant, nausea, and fever – are highly suggestive of this diagnosis.

### Additional Notes:
*   Other possible diagnoses to consider include:
    *   Inflammatory bowel disease (IBD)
    *   Gastroenteritis
    *   Other abdominal pain syndromes
*   It is important to note that this is a preliminary assessment based on the provided information. A definitive diagnosis requires further investigation, such as imaging studies (such as ultrasound or CT scan).


In [41]:
question = """
A 70-year-old man develops:
- resting tremor
- bradykinesia
- muscular rigidity
- shuffling gait

What is the most likely diagnosis and how to do it?
"""

response = generate_medical_response(question)

print(response)

The most likely diagnosis is Parkinson's disease.

Here's how we can approach this:

**1. Clinical Presentation:**

*   **Resting Tremor:** This is a hallmark symptom of Parkinson's disease, characterized by a tremor that is noticeable when the person is at rest.
*   **Bradykinesia:** This refers to slowness of movement. It's a significant feature of Parkinson's, causing difficulty initiating and executing movements.
*   **Muscular Rigidity:** This is an increased resistance to passive movement, causing muscles to contract involuntarily.
*   **Shuffling Gait:** This is a characteristic gait pattern often associated with Parkinson's, reflecting the impaired coordination and movement control.

**2. Diagnostic Considerations:**

*   **Neurological Examination:** A thorough neurological examination is crucial. This includes:
    *   **Muscle Strength:** Assessing strength in different muscle groups to evaluate for weakness.
    *   **Reflexes:** Testing reflexes (e.g., knee-jerk, ankle-jer

In [42]:
question = """
A 28-year-old woman presents with:
- lower abdominal pain
- vaginal bleeding
- missed menstrual period

Pregnancy test is positive.

Which diagnosis must be ruled out immediately?
"""

response = generate_medical_response(question)

print(response)

The most immediate concern is a possible ectopic pregnancy.

### Explanation:

Ectopic pregnancy is a life-threatening condition where the fertilized egg implants outside the uterus. It is a significant concern given the positive pregnancy test and the patient's symptoms.  It requires immediate investigation and treatment to prevent complications like severe bleeding, infection, and preterm labor.

Other possible diagnoses to consider include:

*   Miscarriage: While a miscarriage can occur without a positive pregnancy test, it's important to rule it out.
*   Uterine rupture: This is a serious condition that can occur with pelvic infections or trauma.
*   Pelvic inflammatory disease (PID): Although less likely given the positive pregnancy test, PID should be considered.

The patient's symptoms, particularly the lower abdominal pain, vaginal bleeding, and missed period, strongly suggest a possible ectopic pregnancy. Prompt evaluation and treatment are crucial.


In [43]:
question = """
A 19-year-old woman with type 1 diabetes presents with vomiting,abdominal pain, and rapid breathing.
Laboratory findings show:
- hyperglycemia
- ketonemia
- metabolic acidosis

What is the most likely diagnosis?

Options:
A. Hyperosmolar hyperglycemic state
B. Diabetic ketoacidosis
C. Acute pancreatitis
D. Lactic acidosis
E. Addison disease
"""

response = generate_medical_response(question)

print(response)

B. Diabetic ketoacidosis

### Explanation:

Here's why:

*   **Diabetic Ketoacidosis (DKA):** This is the most likely diagnosis given the patient's presentation. The combination of hyperglycemia, ketonemia (high levels of ketones in the blood), and metabolic acidosis (acid buildup in the blood) are classic signs of DKA.  The patient's symptoms of vomiting, abdominal pain, and rapid breathing are also consistent with the metabolic disturbances associated with DKA.

Let's look at why the other options are less likely:

*   **Hyperosmolar hyperglycemic state (HHS):** While HHS can occur with diabetes, it typically presents with severe dehydration and altered mental status, not the immediate symptoms described.
*   **Acute pancreatitis:** This is a serious condition, but it usually presents with epigastric pain, nausea, and vomiting, not the rapid breathing and ketone levels.
*   **Lactic acidosis:** This is typically caused by a buildup of lactic acid in the blood, often due to muscle bre

In [44]:
question = """
A 58-year-old man comes to the emergency department because of crushing chest pain radiating to his left arm for the past 45 minutes. He is diaphoretic and nauseated. ECG shows ST-segment elevation in leads II, III, and aVF. Blood pressure is 82/58 mmHg. Lungs are clear on auscultation.
What is the most likely diagnosis and the immediate next step in management?
"""

response = generate_medical_response(question)

print(response)

The most likely diagnosis is acute myocardial infarction (AMI), specifically an inferior wall MI.

**Immediate Next Step in Management:**

1.  **Oxygen:** Administer oxygen via nasal cannula to maintain SpO2 > 90%.
2.  **Aspirin:** Administer aspirin 325 mg orally.
3.  **Nitroglycerin:** Administer sublingual nitroglycerin 0.4 mg every 5 minutes for up to 3 doses, if pain persists.
4.  **Morphine:** Administer morphine 2-4 mg IV for pain control.
5.  **Anticoagulation:** Initiate anticoagulation with heparin (unfractionated heparin or low molecular weight heparin) or bivalirudin.
6.  **Cardiac Monitoring:** Continuous cardiac monitoring with ECG.
7.  **IV Access:** Establish IV access.
8.  **Fluid Resuscitation:** Administer IV fluids (e.g., normal saline) to maintain adequate perfusion.
9.  **Consider further investigation:** Obtain cardiac biomarkers (troponin) to confirm the diagnosis and assess for myocardial damage.

**Explanation:**

*   **ST-segment elevation in leads II, III, a

In [45]:
question = """
A newborn fails to pass meconium within the first 48 hours of life and develops abdominal distension with bilious vomiting. Rectal examination causes explosive release of stool.
What is the most likely diagnosis and underlying pathology?
"""

response = generate_medical_response(question)

print(response)

The most likely diagnosis is **intestinal atresia**.

Here's why:

*   **Meconium Failure:** The failure to pass meconium within the first 48 hours is a classic sign of intestinal atresia.
*   **Bilious Vomiting:** This is a significant finding, as it indicates the presence of bile in the stool, which is a hallmark of intestinal atresia.
*   **Explosive Release of Stool:** This is a characteristic finding in atresia, where the bowel is unable to contract properly, leading to a sudden and forceful expulsion of stool.
*   **Abdominal Distension with Bilious Vomiting:** The distension and vomiting are consistent with a bowel obstruction.

**Important Considerations:**

*   **Differential Diagnosis:** While intestinal atresia is the most probable diagnosis, other possibilities should be considered, including:
    *   **Intussusception:** This is a less likely but still possible diagnosis, especially if the infant is young.
    *   **Congenital Obstruction:**  A blockage in the intestines.


In [60]:
question = """
A patient presents with weight loss, heat intolerance, palpitations, and exophthalmos. What is the most likely diagnosis and underlying autoimmune mechanism?
"""

response = generate_medical_response(question)

print(response)

The most likely diagnosis is systemic lupus erythematosus (SLE).

### Explanation:

Here's a breakdown of why this is the most likely diagnosis and the underlying autoimmune mechanism:

*   **Symptoms:** The patient presents with a constellation of symptoms that are highly suggestive of SLE:
    *   **Weight loss:**  A significant and persistent weight loss is a hallmark of SLE.
    *   **Heat intolerance:**  This can be due to inflammation and vascular changes.
    *   **Palpitations:**  These are often associated with vascular changes and inflammation.
    *   **Exophthalmos:**  This is a bulging of the eye, which is a classic sign of SLE and is caused by inflammation of the optic nerve and surrounding structures.

*   **Autoimmune Mechanism:** SLE is an autoimmune disease where the body's immune system mistakenly attacks its own tissues. In SLE, the immune system attacks the endothelium (lining of blood vessels), leading to:
    *   **Vasculitis:** Inflammation of blood vessels, cau

In [48]:
question = """
Is diabetes completely curable?
"""

response = generate_medical_response(question)

print(response)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
**No, diabetes is not completely curable.**

While advancements in treatment have significantly improved the lives of people with diabetes, it is not currently possible to cure the disease.

Here's a more detailed explanation:

*   **Diabetes is a chronic condition:** It's a long-term health issue that requires ongoing management.
*   **Multiple complications:** Diabetes can lead to a wide range of complications, including:
    *   Heart disease
    *   Kidney disease
    *   Nerve damage (neuropathy)
    *   Eye damage (retinopathy)
    *   Foot problems (ulcers, infections)
    *   Increased risk of infections
*   **Management is key:**  The primary focus of treatment is managing blood sugar levels through diet, exercise, and medication.  Regular monitoring and lifestyle changes are crucial.
*   **Research continues:** Ongoing resea

In [49]:
question = """
How can diabetes affect the kidneys if it remains untreated?
"""

response = generate_medical_response(question)

print(response)

Diabetes can significantly affect the kidneys if it remains untreated. Here's a breakdown of how it can occur:

*   **Diabetic Nephropathy:** This is the primary damage to the kidneys caused by diabetes. Over time, high blood sugar levels can damage the small blood vessels in the kidneys, leading to scarring and inflammation.
*   **Glomerular Damage:** The glomeruli are the filtering units of the kidneys. Diabetes can cause the glomeruli to become damaged and less efficient at filtering waste products from the blood.
*   **Proteinuria:**  High blood sugar levels can cause protein to leak into the urine. This is called proteinuria, and it's a sign of kidney damage.
*   **Increased Blood Pressure:** Diabetes often leads to high blood pressure, which further stresses the kidneys and accelerates damage.
*   **Fluid Retention:**  Diabetes can cause the kidneys to retain fluid, leading to swelling in the legs and ankles.

**Important Note:**  Untreated diabetes can lead to serious complicati

In [50]:
question = """
How can someone tell if they have anemia?
"""

response = generate_medical_response(question)

print(response)

Anemia can be detected through a variety of methods, including:

*   **Physical Examination:** A doctor may observe your appearance, looking for signs of pallor (pale skin), shortness of breath, or fatigue.
*   **Blood Tests:**
    *   **Complete Blood Count (CBC):** This is the most common test for anemia. It measures red blood cells, hemoglobin, and white blood cells.
    *   **Iron Studies:** These tests look for low iron levels, including serum iron, ferritin, and transferrin saturation.
    *   **Vitamin B12 and Folate Levels:** These tests can help determine if there's a deficiency.
*   **Symptoms:** Some people experience anemia symptoms such as fatigue, weakness, shortness of breath, dizziness, and pale skin.


In [51]:
question = """
How can I identify food poisoning symptoms?
"""

response = generate_medical_response(question)

print(response)

Okay, let's talk about how to identify food poisoning symptoms. It's a common concern, and understanding the signs can help you take action quickly. Here's a breakdown of common symptoms:

**1. Common Symptoms:**

*   **Nausea:** Feeling like you want to vomit. This is often the first and most noticeable symptom.
*   **Vomiting:** Throwing up. This can be forceful or infrequent.
*   **Diarrhea:** Loose, watery stools. This can be sudden or gradual.
*   **Abdominal Cramps:** Painful spasms in your stomach.
*   **Fever:** A temperature of 100.4°F (38°C) or higher.
*   **Chills:** Feeling cold and shivering.
*   **Headache:** A general ache or pain in your head.
*   **Weakness:** Feeling tired and lacking energy.

**2.  Symptoms That May Indicate a More Serious Condition:**

*   **Bloody Stools:**  Seeing blood in your stool.
*   **Severe Dehydration:**  Signs like decreased urination, dizziness, and extreme thirst.
*   **Persistent Vomiting:** Vomiting that doesn't stop even after you've

In [52]:
question = """
What options exist for the replacement Of Artificial Valves?
"""

response = generate_medical_response(question)

print(response)

The replacement of artificial valves is a complex and evolving field, and there isn't a single, universally agreed-upon "best" option.  The optimal choice depends heavily on the specific valve type, the patient's condition, and the surgeon's expertise.  Here's a breakdown of the current options, categorized by approach:

**1. Mechanical Replacements:**

*   **Mechanical Valves:** These are the most common type of replacement. They are designed to be replaced with a mechanical valve.
    *   **Mechanical Valves with a New Valve:**  These valves are designed to be replaced with a new valve.  This is often the preferred option for patients with significant valve disease.
    *   **Mechanical Valves with a Modified Valve:**  These valves are designed to be replaced with a modified valve, which can be more durable and have a longer lifespan.
*   **Mechanical Valves with a Bioprosthetic Valve:**  These valves are a newer technology offering a more durable and reliable replacement. They are t

In [53]:
question = """
What are the symptoms of cardiac tamponade?
"""

response = generate_medical_response(question)

print(response)

Cardiac tamponade is a life-threatening condition where fluid accumulates in the sac surrounding the heart. This fluid buildup can compress the heart, impairing its ability to pump blood effectively.

Here are the common symptoms of cardiac tamponade:

*   Shortness of breath
*   Chest pain
*   Rapid heart rate
*   Fatigue
*   Dizziness
*   Cyanosis (bluish discoloration of the skin)
*   Low blood pressure


In [54]:
question = """
What to eat at the restaurant to keep body healthy?
"""

response = generate_medical_response(question)

print(response)

Okay, let's talk about what to eat at a restaurant to support your health. It's great that you're focusing on your well-being! Here's a breakdown of foods that can really benefit you:

**1. Lean Proteins:**
   * **Why they're good:** Protein is essential for building and repairing tissues, making enzymes and hormones, and supporting a healthy immune system.
   * **Good choices:** Grilled chicken or fish, lean beef, turkey, tofu, beans, lentils, Greek yogurt.

**2. Fruits & Vegetables:**
   * **Why they're good:** Packed with vitamins, minerals, antioxidants, and fiber. They help protect your cells from damage and keep your body functioning optimally.
   * **Focus on:** A colorful array of fruits and vegetables – berries, leafy greens, broccoli, carrots, peppers, etc.

**3. Whole Grains:**
   * **Why they're good:** Provide sustained energy and fiber, which aids in digestion and helps regulate blood sugar levels.
   * **Good choices:** Brown rice, quinoa, whole-wheat bread, oats.

**4. 

In [55]:
raise SystemExit("Stopping notebook here")

SystemExit: Stopping notebook here

##  **9.  Pushing Model for Deployment**

In [57]:
hf_token = UserSecretsClient().get_secret("keyhf") 
login(token=hf_token)

In [58]:
# 2. Push to the hub (adds safetensors, configs, and vocab files)
model.push_to_hub("VPrerana/gemma-3-1b-Medical-it")
tokenizer.push_to_hub("VPrerana/gemma-3-1b-Medical-it")

README.md:   0%|          | 0.00/544 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/VPrerana/gemma-3-1b-Medical-it


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

#### **To use the model click to below link -**

*https://huggingface.co/VPrerana/gemma-3-1b-Medical-it*

#### **To use dataset click below link -**

*https://huggingface.co/datasets/VPrerana/medical_meadow_wikidoc_patient_information*